In [137]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler ,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

In [138]:
df=pd.read_csv("../data/cleaned_cpu_scheduling_process.csv")
df.head()

,arrival_time,burst_time,priority,process_type,best_algorithm,burst_priority_ratio,interaction_1,interaction_2
0,0,1,5,IO-bound,SJF,0.166667,5,0
1,20,14,9,IO-bound,Round Robin,1.400000,126,280
2,11,15,8,IO-bound,Priority,1.666667,120,165
3,4,18,7,IO-bound,Round Robin,2.250000,126,72
4,20,13,5,CPU-bound,Round Robin,2.166667,65,260


In [ ]:
df["relative_burst"] = df["burst_time"] / (df["burst_priority_ratio"] + 1)
df["arrival_burst_ratio"] = df["arrival_time"] / (df["burst_time"] + 1)
df["priority_burst_interaction"] = df["priority"] / (df["burst_time"] + 1)
df["soft_short"] = (df["burst_time"] < 7).astype(int)
df["arrival_stability"] = df["arrival_time"] / (df["burst_time"] + 1)
df["sjf_vs_srtf_signal"] = df["burst_time"] / (df["arrival_time"] + 1)
df['interaction_1'] = df['burst_time'] * df['priority']
df['interaction_2'] = df['arrival_time'] * df['burst_time']
df['burst_priority_ratio'] = df['burst_time'] / (df['priority'] + 1)




In [140]:
df['best_algorithm'].value_counts()

best_algorithm
Round Robin    1548
FCFS           1352
Priority       1185
SJF             646
SRTF            269
Name: count, dtype: int64

In [ ]:
features = [
    "arrival_time",
    "burst_time",
    "priority",
    "burst_priority_ratio",
    "interaction_1",
    "interaction_2",
    "process_type",
    "soft_short",
    "arrival_stability",
    "relative_burst",
    "arrival_burst_ratio",
    "priority_burst_interaction",
    "sjf_vs_srtf_signal"
]

X = df[features]

# Encode target
le = LabelEncoder()
y = le.fit_transform(df["best_algorithm"])

In [142]:
numerical_features = [
    "arrival_time",
    "burst_time",
    "priority",
    "burst_priority_ratio",
    "interaction_1",
    "interaction_2",
    "arrival_stability",
    "relative_burst",
    "arrival_burst_ratio",
    "priority_burst_interaction",
    "soft_short",
    "sjf_vs_srtf_signal"
]

categorical_features = ["process_type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [143]:
models = {
    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=10),
        n_estimators=150,
        random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2),
        n_estimators=200,
        learning_rate=0.1,
        algorithm="SAMME",   # fixes warning
        random_state=42
    ),

    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    )
}

In [144]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [145]:
results = {}

for name, model in models.items():

    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    print(f"\n{name}")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    results[name] = acc


Bagging
Accuracy: 0.6120
              precision    recall  f1-score   support

        FCFS       0.59      0.59      0.59       270
    Priority       0.71      0.84      0.77       237
 Round Robin       0.55      0.52      0.54       310
         SJF       0.59      0.33      0.43       129
        SRTF       0.61      0.93      0.74        54

    accuracy                           0.61      1000
   macro avg       0.61      0.64      0.61      1000
weighted avg       0.61      0.61      0.60      1000


AdaBoost
Accuracy: 0.6050
              precision    recall  f1-score   support

        FCFS       0.61      0.54      0.57       270
    Priority       0.71      0.88      0.78       237
 Round Robin       0.58      0.55      0.56       310
         SJF       0.44      0.45      0.44       129
        SRTF       0.57      0.43      0.49        54

    accuracy                           0.60      1000
   macro avg       0.58      0.57      0.57      1000
weighted avg       0.60 

In [146]:
best_model_name = max(results, key=results.get)

print("\nBest Model:", best_model_name)
print("Best Accuracy:", results[best_model_name])


Best Model: Bagging
Best Accuracy: 0.612


In [147]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

bag_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    random_state=42
)

param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__estimator__max_depth": [5, 10, 15],
    "model__max_samples": [0.6, 0.8, 1.0],
    "model__max_features": [0.6, 0.8, 1.0]
}

In [148]:
pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", bag_model)
])

In [149]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('num',
                                                                               StandardScaler(),
                                                                               ['arrival_time',
                                                                                'burst_time',
                                                                                'priority',
                                                                                'burst_priority_ratio',
                                                                                'interaction_1',
                                                                                'interaction_2',
                                                                                'arrival_stability',
                                                                                'relative_burst',
                                                                                'arrival_burst_ratio',
                                                                                'priority_burst_interaction',
                                                                                'soft_short',
                                                                                'sjf_vs_srtf_signal']),
                                                                              ('cat',
                                                                               O...oder(handle_unknown='ignore'),
                                                                               ['process_type'])])),
                                             ('model',
                                              BaggingClassifier(estimator=DecisionTreeClassifier(),
                                                                random_state=42))]),
                   n_jobs=-1,
                   param_distributions={'model__estimator__max_depth': [5, 10,
                                                                        15],
                                        'model__max_features': [0.6, 0.8, 1.0],
                                        'model__max_samples': [0.6, 0.8, 1.0],
                                        'model__n_estimators': [100, 200, 300]},
                   random_state=42, scoring='f1_weighted')

In [150]:
print("Best Params:", random_search.best_params_)
print("Best CV Score (F1):", random_search.best_score_)

Best Params: {'model__n_estimators': 100, 'model__max_samples': 0.8, 'model__max_features': 0.8, 'model__estimator__max_depth': 5}
Best CV Score (F1): 0.6230990260845966


In [151]:
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report

print("Test Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Test Accuracy: 0.634

Classification Report:

              precision    recall  f1-score   support

        FCFS       0.61      0.58      0.59       270
    Priority       0.71      0.88      0.78       237
 Round Robin       0.59      0.53      0.56       310
         SJF       0.60      0.39      0.47       129
        SRTF       0.63      1.00      0.77        54

    accuracy                           0.63      1000
   macro avg       0.63      0.68      0.64      1000
weighted avg       0.63      0.63      0.62      1000



In [152]:
results["Bagging_Tuned"] = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "F1 Score": random_search.best_score_
}

In [153]:
df.columns

Index(['arrival_time', 'burst_time', 'priority', 'process_type',
       'best_algorithm', 'burst_priority_ratio', 'interaction_1',
       'interaction_2', 'relative_burst', 'arrival_burst_ratio',
       'priority_burst_interaction', 'soft_short', 'arrival_stability',
       'sjf_vs_srtf_signal'],
      dtype='object')